In [ ]:
import json
import pandas as pd
import h5py

# ============================
# PATH DATA
# ============================

JSON_PATH = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/radar_inventory_3ch_detailed.json"
STEAD_CSV = "/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv"
STEAD_HDF5 = "/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5"

# ============================
# 1. BACA JSON → AMBIL KODE STASIUN
# ============================

with open(JSON_PATH, "r") as f:
    radar_data = json.load(f)

station_codes = set()

def extract_station_codes(obj):
    """Ambil semua kode stasiun dari JSON (rekursif)."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k.lower() in ["station", "sta", "station_code", "code"]:
                station_codes.add(str(v).upper())
            extract_station_codes(v)
    elif isinstance(obj, list):
        for item in obj:
            extract_station_codes(item)

extract_station_codes(radar_data)

print("📌 Total stasiun ditemukan di JSON:", len(station_codes))
print("Contoh:", list(station_codes)[:10])

# ============================
# 2. BACA STEAD CSV
# ============================

df = pd.read_csv(STEAD_CSV)

# STEAD pakai kolom receiver_code untuk kode stasiun
df["receiver_code"] = df["receiver_code"].astype(str).str.upper()

print("📖 Total data STEAD:", len(df))

# ============================
# 3. FILTER STEAD BERDASARKAN STASIUN JSON
# ============================

df_filtered = df[df["receiver_code"].isin(station_codes)]
print("🎯 Total data STEAD hasil filter:", len(df_filtered))

df_filtered.to_csv("stead_filtered_by_station.csv", index=False)
print("💾 Disimpan ke stead_filtered_by_station.csv")

# ============================
# 4. OPSIONAL: AMBIL WAVEFORM DARI HDF5
# ============================

def load_waveform(trace_name, hdf5_path=STEAD_HDF5):
    with h5py.File(hdf5_path, "r") as f:
        if trace_name in f:
            return f[trace_name][:]
        return None

# Contoh ambil waveform pertama
if len(df_filtered) > 0:
    sample_id = df_filtered.iloc[0]["trace_name"]
    waveform = load_waveform(sample_id)
    print("📈 Waveform shape:", waveform.shape if waveform is not None else "Tidak ditemukan")


In [ ]:
import pandas as pd
from datetime import timedelta

# ============================
# PATH DATA
# ============================

STEAD_FILTERED = "stead_filtered_by_station.csv"
BMKG_CATALOG = "/Volumes/Extreme SSD/Indonesian Earthquake Catalog (BMKG), 1998–2024/BMKG_Earthquake_Catalog.csv"

# ============================
# 1. LOAD DATA
# ============================

df_stead = pd.read_csv(STEAD_FILTERED)
df_bmkg = pd.read_csv(BMKG_CATALOG)

# ============================
# 2. NORMALISASI WAKTU BMKG
# ============================

# Gabungkan Date + Time (UTC)
df_bmkg["origin_time"] = pd.to_datetime(
    df_bmkg["Date"] + " " + df_bmkg["Time (UTC)"],
    errors="coerce"
)

# Normalisasi waktu STEAD
df_stead["origin_time"] = pd.to_datetime(df_stead["source_origin_time"], errors="coerce")

print("Total STEAD:", len(df_stead))
print("Total BMKG:", len(df_bmkg))

# ============================
# 3. MATCHING EVENT
# ============================

matches = []

for _, s in df_stead.iterrows():
    t0 = s["origin_time"]

    # Cari event BMKG dalam ±5 detik
    df_time = df_bmkg[
        (df_bmkg["origin_time"] >= t0 - timedelta(seconds=5)) &
        (df_bmkg["origin_time"] <= t0 + timedelta(seconds=5))
    ]

    if len(df_time) == 0:
        continue

    # Tambahkan info STEAD
    df_time = df_time.copy()
    df_time["stead_trace"] = s["trace_name"]
    df_time["stead_station"] = s["receiver_code"]
    df_time["stead_lat"] = s["source_latitude"]
    df_time["stead_lon"] = s["source_longitude"]
    df_time["stead_mag"] = s["source_magnitude"]

    matches.append(df_time)

# ============================
# 4. SIMPAN HASIL
# ============================

if len(matches) > 0:
    df_match = pd.concat(matches, ignore_index=True)
    df_match.to_csv("matched_stead_bmkg_events.csv", index=False)
    print("🎯 Event cocok ditemukan:", len(df_match))
else:
    print("❌ Tidak ada event STEAD yang cocok dengan katalog BMKG.")


KeyError: 'Origin Time (UTC)'

In [5]:
import pandas as pd

BMKG_CATALOG = "/Volumes/Extreme SSD/Indonesian Earthquake Catalog (BMKG), 1998–2024/BMKG_Earthquake_Catalog.csv"

df_bmkg = pd.read_csv(BMKG_CATALOG)
print(df_bmkg.columns)


Index(['No', 'Event ID', 'Unnamed: 2', 'Date', 'Time (UTC)', 'Latitude',
       'Longitude', 'Magnitude', 'Mag Type', 'Depth (km)', 'Source',
       'Source Event ID'],
      dtype='object')
